In [1]:
import pandas as pd
import numpy as np


# ==============================
# 1. Load original dataset
# ==============================

df = pd.read_csv("commodity_futures.csv")


# Date conversion
df["Date"] = pd.to_datetime(
    df["Date"],
    dayfirst=True,
    errors="coerce"
)


# Remove bad dates
df = df.dropna(subset=["Date"])


# Sort by date
df = df.sort_values("Date")


# ==============================
# 2. Select required products
# ==============================

products = [
    "COFFEE",
    "SUGAR",
    "SOYBEANS",
    "WHEAT",
    "COTTON"
]


df = df[
    [
        "Date"
    ] + products
]


# ==============================
# 3. Clean missing values
# ==============================

for col in products:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    df[col] = df[col].interpolate()


print("Historical data:")
print(df.head())

print(df.tail())


# ==============================
# 4. Create future dates
# ==============================

last_date = df["Date"].max()


future_dates = pd.date_range(
    start=last_date + pd.Timedelta(days=1),
    end="2026-12-31",
    freq="D"
)


future = pd.DataFrame(
    {
        "Date":future_dates
    }
)


# ==============================
# 5. Synthetic generation
# ==============================


np.random.seed(42)


for product in products:


    # last known price

    last_price = df[product].iloc[-1]


    days = len(future)


    # -----------------------
    # Trend component
    # -----------------------

    trend_strength = np.random.uniform(
        -0.0001,
        0.0003
    )


    trend = (
        last_price *
        (1 + trend_strength*np.arange(days))
    )


    # -----------------------
    # Seasonality component
    # -----------------------

    seasonal = (
        np.sin(
            np.arange(days)
            *
            2*np.pi/365
        )
        *
        last_price
        *
        0.03
    )


    # -----------------------
    # Market volatility
    # -----------------------

    volatility = np.random.normal(
        loc=0,
        scale=last_price*0.02,
        size=days
    )


    # -----------------------
    # Final price
    # -----------------------

    generated = (
        trend
        +
        seasonal
        +
        volatility
    )


    # avoid negative prices

    generated = np.maximum(
        generated,
        0
    )


    future[product] = generated



# ==============================
# 6. Combine historical + future
# ==============================

final_df = pd.concat(
    [
        df,
        future
    ],
    ignore_index=True
)


# remove duplicates

final_df = final_df.drop_duplicates(
    subset=["Date"]
)


final_df = final_df.sort_values(
    "Date"
)


# ==============================
# 7. Save file
# ==============================

final_df.to_csv(
    "synthetic_commodity_2026.csv",
    index=False
)


# ==============================
# 8. Check
# ==============================

print("\nFinal shape:")
print(final_df.shape)


print("\nLast rows:")
print(final_df.tail())


Historical data:
          Date  COFFEE  SUGAR  SOYBEANS   WHEAT  COTTON
21  2000-01-02  111.65   5.39     510.0  258.00   57.13
42  2000-01-03  101.15   5.08     496.5  247.75   59.50
85  2000-01-05  100.30   6.53     555.0  251.00   56.30
108 2000-01-06   94.45   7.79     522.5  270.50   60.25
151 2000-01-08   86.30  10.35     442.0  245.00   59.01
           Date  COFFEE  SUGAR  SOYBEANS    WHEAT  COTTON
5945 2023-12-01  149.40  19.59   1529.50  742.750   82.04
6009 2023-12-04  190.25  24.05   1504.25  679.500   82.45
6031 2023-12-05  186.75  26.20   1435.00  656.625   80.38
6052 2023-12-06  185.55  25.47   1372.75  633.750   83.49
6074 2023-12-07  158.75  23.91   1488.50  621.250   82.51

Final shape:
(3523, 6)

Last rows:
           Date      COFFEE      SUGAR     SOYBEANS       WHEAT     COTTON
3518 2026-12-27  170.304078  29.788931  1917.325216  643.676712  81.021322
3519 2026-12-28  163.910849  30.055638  1880.667401  648.861696  79.567661
3520 2026-12-29  166.379813  29.446731